# 01 – Embed PDF Documents and Save FAISS Vector Store

This notebook demonstrates how to:
1. Load one or more PDF documents
2. Split them into chunks
3. Embed the chunks using **Cohere Embeddings**
4. Store the resulting vectors in a **FAISS** index and persist it to disk

> **Prerequisites**  
> - Set the environment variable `COHERE_API_KEY` (or place it in a `.env` file in the repo root).  
> - Put the PDF files you want to embed in the `./docs/` folder.

## 1. Imports and configuration

In [1]:
import os
from pathlib import Path

from dotenv import load_dotenv
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.vectorstores import FAISS
from langchain_cohere import CohereEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Load environment variables from .env if present
load_dotenv()

# --- Configuration -----------------------------------------------------------
DOCS_DIR       = Path("./docs")          # folder containing your PDF files
FAISS_INDEX    = Path("./faiss_index")   # where to save the FAISS index

# Cohere embedding model (most current as of 2025)
EMBED_MODEL    = "embed-english-v3.0"

# Text splitting parameters
CHUNK_SIZE     = 1000
CHUNK_OVERLAP  = 200
# -----------------------------------------------------------------------------

cohere_api_key = os.environ.get("COHERE_API_KEY")
if not cohere_api_key:
    raise EnvironmentError(
        "COHERE_API_KEY is not set. "
        "Add it to your environment or create a .env file with COHERE_API_KEY=<your-key>."
    )

print("✅ Configuration OK")

/tmp/ipykernel_5076/4142412935.py:5: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


✅ Configuration OK


## 2. Load PDF documents

In [2]:
if not DOCS_DIR.exists():
    DOCS_DIR.mkdir(parents=True)
    print(f"Created '{DOCS_DIR}' – place your PDF files there and re-run this cell.")
else:
    pdf_files = sorted(DOCS_DIR.glob("*.pdf"))
    print(f"Found {len(pdf_files)} PDF file(s) in '{DOCS_DIR}':")
    for p in pdf_files:
        print(f"  • {p.name}")

Found 1 PDF file(s) in 'docs':
  • Good_practice_CLTS_Uganda.pdf


In [3]:
raw_documents = []
for pdf_path in sorted(DOCS_DIR.glob("*.pdf")):
    loader = PyPDFLoader(str(pdf_path))
    pages = loader.load()
    raw_documents.extend(pages)
    print(f"  Loaded '{pdf_path.name}': {len(pages)} page(s)")

print(f"\n📄 Total pages loaded: {len(raw_documents)}")

  Loaded 'Good_practice_CLTS_Uganda.pdf': 24 page(s)

📄 Total pages loaded: 24


## 3. Split documents into chunks

In [4]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
)

documents = text_splitter.split_documents(raw_documents)
print(f"✂️  Split into {len(documents)} chunk(s)")

# Preview first chunk
if documents:
    print("\n--- Preview of first chunk ---")
    print(documents[0].page_content[:500])
    print("\nMetadata:", documents[0].metadata)

✂️  Split into 75 chunk(s)

--- Preview of first chunk ---
GOOD PRACTICES IN COMMUNITY-LED TOTAL SANITATION  
Plan’s experience in Uganda 2007 - 2010
May 2011

Metadata: {'producer': 'PyPDF', 'creator': 'PyPDF', 'creationdate': '2012-03-12T12:00:48+03:00', 'moddate': '2012-03-17T20:41:38+03:00', 'source': 'docs/Good_practice_CLTS_Uganda.pdf', 'total_pages': 24, 'page': 0, 'page_label': '1'}


## 4. Create Cohere embeddings

In [5]:
embeddings = CohereEmbeddings(
    model=EMBED_MODEL,
    cohere_api_key=cohere_api_key,
)

# Quick sanity check – embed a single test string
test_vector = embeddings.embed_query("Hello, Cohere!")
print(f"🔢 Embedding dimension: {len(test_vector)}")

🔢 Embedding dimension: 1024


## 5. Build FAISS index and persist to disk

In [6]:
print("Building FAISS index – this may take a moment for large document sets...")

vector_store = FAISS.from_documents(documents, embeddings)

vector_store.save_local(str(FAISS_INDEX))
print(f"\n💾 FAISS index saved to '{FAISS_INDEX}'")
print(f"   Files: {[p.name for p in FAISS_INDEX.iterdir()]}")

Building FAISS index – this may take a moment for large document sets...

💾 FAISS index saved to 'faiss_index'
   Files: ['index.pkl', 'index.faiss']


## 6. Verify – reload the index and run a test query

In [7]:
# Reload from disk to verify everything was saved correctly
loaded_store = FAISS.load_local(
    str(FAISS_INDEX),
    embeddings,
    allow_dangerous_deserialization=True,
)

test_query = "What is this document about?"
results = loaded_store.similarity_search(test_query, k=3)

print(f"🔍 Top {len(results)} result(s) for: '{test_query}'\n")
for i, doc in enumerate(results, 1):
    print(f"Result {i} | Source: {doc.metadata.get('source', 'unknown')} | Page: {doc.metadata.get('page', '?')}")
    print(doc.page_content[:300])
    print("-" * 60)

🔍 Top 3 result(s) for: 'What is this document about?'

Result 1 | Source: docs/Good_practice_CLTS_Uganda.pdf | Page: 7
staff on how to apply CLTS and support its 
monitoring. 
The World Bank Water and Sanitation 
Programme (WSP) in Uganda working closely 
with Plan (U) and other members of the 
National Sanitation Working group (NSWG) has 
commissioned the development of National 
CLTS training manuals which will ac
------------------------------------------------------------
Result 2 | Source: docs/Good_practice_CLTS_Uganda.pdf | Page: 4
and, to date, more than 50 villages supported by 
Plan have been declared Open Defecation Free 
(ODF) – where every household uses a latrine as a 
safe method of excreta disposal.  
There was no documentation of good practices 
and lessons learnt in the application of CLTS 
among these communities. 
------------------------------------------------------------
Result 3 | Source: docs/Good_practice_CLTS_Uganda.pdf | Page: 6
well as the protection of foo

---
✅ **Done!** The FAISS index is saved to `./faiss_index/`.  
Open **`02_retrieve_rerank_generate.ipynb`** to query it.